# SETUP

In [3]:
# SETUP

!pip install nibabel monai

import os
import numpy as np
import torch
from monai.data import Dataset, DataLoader
from monai.transforms import *
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# DATA SPLIT (TRAIN / VAL)

In [4]:
# SPLIT

DATA_PATH = "/kaggle/working/brats_clean"

patients = sorted([
    os.path.join(DATA_PATH, p)
    for p in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, p))
])

# split 80/20
split = int(0.8 * len(patients))

train_patients = patients[:split]
val_patients = patients[split:]

print(len(train_patients), len(val_patients))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/brats_clean'

CREAR DICCIONARIOS (MONAI STYLE)

In [ ]:
def create_dicts(patient_list):
    data = []
    for p in patient_list:
        pid = os.path.basename(p)

        data.append({
            "image": [
                os.path.join(p, f"{pid}_flair.nii.gz"),
                os.path.join(p, f"{pid}_t1.nii.gz"),
                os.path.join(p, f"{pid}_t1ce.nii.gz"),
                os.path.join(p, f"{pid}_t2.nii.gz"),
            ],
            "label": os.path.join(p, f"{pid}_seg.nii.gz")
        })
    return data

train_files = create_dicts(train_patients)
val_files = create_dicts(val_patients)

# TRANSFORMS

In [ ]:
# TRANSFORMS

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),

    # Convertimos labels 1,2,4 → 3 canales
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),

    Spacingd(keys=["image", "label"], pixdim=(1.0,1.0,1.0), mode=("bilinear","nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),

    RandSpatialCropd(keys=["image", "label"], roi_size=(128,128,128), random_size=False),

    # Augmentations 3D
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),

    RandRotate90d(keys=["image", "label"], prob=0.5),

    RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),

    EnsureTyped(keys=["image", "label"]),
])

# VALIDATION TRANSFORMS

In [ ]:
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
])

# DATASETS + LOADERS

In [ ]:
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1)

# MODELO (U-NET MONAI)

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=3,  # WT, TC, ET
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# LOSS + MÉTRICAS

In [ ]:
loss_function = DiceLoss(sigmoid=True)

dice_metric = DiceMetric(include_background=False, reduction="mean")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TRAIN + VALIDATION LOOP

In [ ]:
max_epochs = 10

for epoch in range(max_epochs):
    model.train()
    train_loss = 0

    for batch in train_loader:
        inputs = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(inputs)
        loss = loss_function(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} Train Loss: {train_loss/len(train_loader):.4f}")

    # VALIDACIÓN
    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for batch in val_loader:
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(inputs)
            outputs = torch.sigmoid(outputs)

            dice_metric(y_pred=outputs, y=labels)

    metric = dice_metric.aggregate().item()
    dice_metric.reset()

    print(f"Validation Dice: {metric:.4f}")

# INFERENCIA SOBRE VOLUMEN COMPLETO

In [ ]:
model.eval()

with torch.no_grad():
    sample = val_ds[0]

    image = sample["image"].unsqueeze(0).to(device)

    pred = sliding_window_inference(
        image,
        roi_size=(128,128,128),
        sw_batch_size=1,
        predictor=model
    )

    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()